In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. CONFIGURAÇÕES DA SIMULAÇÃO
# ============================================================

a = 0.8                                # Base das sequências exponenciais a^n
W0 = np.pi / 4                         # Frequência das sequências senoidais (rad/amostra)

NM = 400                               # Termos usados nas somas diretas da definição
n_dir = np.arange(-NM, NM + 1)         # Índices da soma direta

n_plot = np.arange(-8, 21)             # Índices para os gráficos das sequências
M_MAX = 60                             # Número máximo de termos nas somas parciais
N_RESP = 60                            # Amostras das respostas ao impulso

# Sistema discreto de exemplo:
# H(z) = (1 + 0.5 z^-1) / (1 - 0.9 z^-1 + 0.81 z^-2)
num_ex = np.array([1.0, 0.5])
den_ex = np.array([1.0, -0.9, 0.81])


# ============================================================
# 2. TRANSFORMADA-Z PELA DEFINIÇÃO
# ============================================================

def transformada_z_direta(x, n, z):
    """
    Calcula a Transformada-Z diretamente pela definição:

    X(z) = sum( x[n] * z^(-n) )

    x : valores da sequência
    n : índices correspondentes
    z : ponto (ou vetor de pontos) do plano complexo
    """

    z = np.atleast_1d(np.asarray(z, dtype=complex))

    potencias = z[None, :] ** (-n[:, None].astype(float))

    return np.sum(x[:, None] * potencias, axis=0)


# ============================================================
# 3. SEQUÊNCIAS ELEMENTARES E SUAS TRANSFORMADAS
# ============================================================

def u(n):
    """
    Degrau unitário u[n].
    """

    return (n >= 0).astype(float)


def x_finita(n):
    """
    Sequência de duração finita {1, 2, 3, 2, 1} para n = 0..4.
    """

    valores = np.array([1.0, 2.0, 3.0, 2.0, 1.0, 0.0])

    return np.where((n >= 0) & (n <= 4), valores[np.clip(n, 0, 5)], 0.0)


cos_w0 = np.cos(W0)
sen_w0 = np.sin(W0)

elementares = [
    {
        "nome": "delta[n]",
        "x": lambda n: (n == 0).astype(float),
        "X": lambda z: np.ones_like(z),
        "Xtxt": "1",
        "roc": "todo z",
        "z0": 0.7 * np.exp(0.9j),
        "tex_x": r"$\delta[n]$",
        "tex_X": r"$X(z)=1$",
    },
    {
        "nome": "delta[n-3]",
        "x": lambda n: (n == 3).astype(float),
        "X": lambda z: z ** (-3),
        "Xtxt": "z^-3",
        "roc": "z != 0",
        "z0": 0.7 * np.exp(0.9j),
        "tex_x": r"$\delta[n-3]$",
        "tex_X": r"$X(z)=z^{-3}$",
    },
    {
        "nome": "u[n]",
        "x": lambda n: u(n),
        "X": lambda z: 1 / (1 - z ** (-1)),
        "Xtxt": "1 / (1 - z^-1)",
        "roc": "|z| > 1",
        "z0": 1.3 * np.exp(0.7j),
        "tex_x": r"$u[n]$",
        "tex_X": r"$X(z)=\frac{1}{1-z^{-1}}$",
    },
    {
        "nome": "a^n u[n]",
        "x": lambda n: np.where(n >= 0, a ** np.clip(n, 0, None), 0.0),
        "X": lambda z: 1 / (1 - a * z ** (-1)),
        "Xtxt": "1 / (1 - a z^-1)",
        "roc": "|z| > |a|",
        "z0": 1.3 * np.exp(0.7j),
        "tex_x": r"$0.8^n\,u[n]$",
        "tex_X": r"$X(z)=\frac{1}{1-0.8z^{-1}}$",
    },
    {
        "nome": "-a^n u[-n-1]",
        "x": lambda n: np.where(n <= -1, -(a ** np.clip(n, None, -1).astype(float)), 0.0),
        "X": lambda z: 1 / (1 - a * z ** (-1)),
        "Xtxt": "1 / (1 - a z^-1)",
        "roc": "|z| < |a|",
        "z0": 0.5 * np.exp(0.7j),
        "tex_x": r"$-0.8^n\,u[-n-1]$",
        "tex_X": r"$X(z)=\frac{1}{1-0.8z^{-1}}$",
    },
    {
        "nome": "n a^n u[n]",
        "x": lambda n: np.where(n >= 0, n * a ** np.clip(n, 0, None), 0.0),
        "X": lambda z: a * z ** (-1) / (1 - a * z ** (-1)) ** 2,
        "Xtxt": "a z^-1 / (1 - a z^-1)^2",
        "roc": "|z| > |a|",
        "z0": 1.3 * np.exp(0.7j),
        "tex_x": r"$n\,0.8^n\,u[n]$",
        "tex_X": r"$X(z)=\frac{0.8z^{-1}}{(1-0.8z^{-1})^2}$",
    },
    {
        "nome": "cos(w0 n) u[n]",
        "x": lambda n: np.where(n >= 0, np.cos(W0 * n), 0.0),
        "X": lambda z: (1 - cos_w0 * z ** (-1)) / (1 - 2 * cos_w0 * z ** (-1) + z ** (-2)),
        "Xtxt": "(1 - cos(w0) z^-1) / (1 - 2cos(w0) z^-1 + z^-2)",
        "roc": "|z| > 1",
        "z0": 1.3 * np.exp(0.7j),
        "tex_x": r"$\cos(\omega_0 n)\,u[n]$",
        "tex_X": r"$X(z)=\frac{1-\cos\omega_0\,z^{-1}}{1-2\cos\omega_0\,z^{-1}+z^{-2}}$",
    },
    {
        "nome": "sen(w0 n) u[n]",
        "x": lambda n: np.where(n >= 0, np.sin(W0 * n), 0.0),
        "X": lambda z: sen_w0 * z ** (-1) / (1 - 2 * cos_w0 * z ** (-1) + z ** (-2)),
        "Xtxt": "sen(w0) z^-1 / (1 - 2cos(w0) z^-1 + z^-2)",
        "roc": "|z| > 1",
        "z0": 1.3 * np.exp(0.7j),
        "tex_x": r"$\sin(\omega_0 n)\,u[n]$",
        "tex_X": r"$X(z)=\frac{\sin\omega_0\,z^{-1}}{1-2\cos\omega_0\,z^{-1}+z^{-2}}$",
    },
    {
        "nome": "a^n cos(w0 n) u[n]",
        "x": lambda n: np.where(n >= 0, a ** np.clip(n, 0, None) * np.cos(W0 * n), 0.0),
        "X": lambda z: (1 - a * cos_w0 * z ** (-1)) / (1 - 2 * a * cos_w0 * z ** (-1) + a ** 2 * z ** (-2)),
        "Xtxt": "(1 - a cos(w0) z^-1) / (1 - 2a cos(w0) z^-1 + a^2 z^-2)",
        "roc": "|z| > |a|",
        "z0": 1.3 * np.exp(0.7j),
        "tex_x": r"$0.8^n\cos(\omega_0 n)\,u[n]$",
        "tex_X": r"$X(z)=\frac{1-a\cos\omega_0\,z^{-1}}{1-2a\cos\omega_0\,z^{-1}+a^2z^{-2}}$",
    },
    {
        "nome": "{1,2,3,2,1}, n=0..4",
        "x": x_finita,
        "X": lambda z: 1 + 2 * z ** (-1) + 3 * z ** (-2) + 2 * z ** (-3) + z ** (-4),
        "Xtxt": "1 + 2z^-1 + 3z^-2 + 2z^-3 + z^-4",
        "roc": "z != 0",
        "z0": 0.5 * np.exp(0.4j),
        "tex_x": r"$\{1,2,3,2,1\}$",
        "tex_X": r"$X(z)=1+2z^{-1}+3z^{-2}+2z^{-3}+z^{-4}$",
    },
]


# ============================================================
# 4. SISTEMAS DISCRETOS: DIFERENÇAS, POLOS, ZEROS E h[n]
# ============================================================

def filtrar(num, den, x):
    """
    Resolve a equação de diferenças recursivamente:

    y[n] = (1/a0) * ( sum(b_k x[n-k]) - sum(a_k y[n-k], k >= 1) )

    que corresponde a H(z) = sum(b_k z^-k) / sum(a_k z^-k)
    """

    y = np.zeros(len(x))

    for i in range(len(x)):
        acumulado = 0.0

        for k, bk in enumerate(num):
            if i - k >= 0:
                acumulado += bk * x[i - k]

        for k in range(1, len(den)):
            if i - k >= 0:
                acumulado -= den[k] * y[i - k]

        y[i] = acumulado / den[0]

    return y


def impulso(N):
    """
    Impulso unitário delta[n] com N amostras.
    """

    x = np.zeros(N)
    x[0] = 1.0

    return x


def polos_e_zeros(num, den):
    """
    Polos e zeros de H(z), multiplicando numerador e denominador
    por z^N (N = maior grau em z^-1):

    H(z) = (b0 z^N + b1 z^(N-1) + ...) / (a0 z^N + a1 z^(N-1) + ...)
    """

    N = max(len(num), len(den)) - 1

    Bz = np.pad(num, (0, N + 1 - len(num)))
    Az = np.pad(den, (0, N + 1 - len(den)))

    return np.roots(Az), np.roots(Bz)


def h_analitico(n, num, den):
    """
    Transformada-Z inversa por frações parciais (polos simples e
    sistema causal):

    H(z) = d + sum( c_k / (1 - p_k z^-1) )
    h[n] = d*delta[n] + sum( c_k * p_k^n ) u[n]
    """

    N = max(len(num), len(den)) - 1

    Bz = np.pad(num, (0, N + 1 - len(num)))
    Az = np.pad(den, (0, N + 1 - len(den)))

    polos = np.roots(Az)

    c = np.polyval(Bz, polos) / (polos * np.polyval(np.polyder(Az), polos))
    d = Bz[0] / Az[0] - np.sum(c)

    h = d * (n == 0).astype(complex)

    for pk, ck in zip(polos, c):
        h += ck * (pk ** n.astype(float)) * (n >= 0)

    return np.real(h)


def resposta_em_frequencia(num, den, omega):
    """
    Avalia H(z) sobre o círculo unitário: H(e^(j*omega)).
    """

    x = np.exp(-1j * omega)

    return np.polyval(num[::-1], x) / np.polyval(den[::-1], x)


def classificar_estabilidade(polos, tol=1e-9):
    """
    Sistema causal: estável se todos os polos estão dentro do
    círculo unitário (a ROC |z| > max|p_k| contém |z| = 1).
    """

    raio_maximo = np.max(np.abs(polos))

    if raio_maximo > 1 + tol:
        return "INSTÁVEL"

    if raio_maximo >= 1 - tol:
        return "MARGINAL"

    return "ESTÁVEL"


# ============================================================
# 5. FUNÇÃO AUXILIAR: PLANO-z COM REGIÃO DE CONVERGÊNCIA
# ============================================================

def desenhar_plano_z(eixo, polos, zeros, regiao, titulo, limite=2.6):
    """
    Desenha o plano-z com círculo unitário, polos, zeros e a ROC
    (regiao é uma função do módulo |z| que devolve True dentro da ROC).
    """

    g = np.linspace(-limite, limite, 600)
    Xg, Yg = np.meshgrid(g, g)
    modulo = np.hypot(Xg, Yg)

    eixo.contourf(Xg, Yg, regiao(modulo).astype(float),
                  levels=[0.5, 1.5], colors=["tab:blue"], alpha=0.25)

    theta = np.linspace(0, 2 * np.pi, 400)
    eixo.plot(np.cos(theta), np.sin(theta), "k--", linewidth=1, label="|z| = 1")

    eixo.plot(np.real(polos), np.imag(polos), "x", color="tab:red",
              markersize=10, markeredgewidth=2.5, label="Polos")

    if len(zeros) > 0:
        eixo.plot(np.real(zeros), np.imag(zeros), "o", color="tab:green",
                  markersize=9, markerfacecolor="none", markeredgewidth=2, label="Zeros")

    eixo.axhline(0, color="k", linewidth=0.8)
    eixo.axvline(0, color="k", linewidth=0.8)

    eixo.set_aspect("equal")
    eixo.set_xlim(-limite, limite)
    eixo.set_ylim(-limite, limite)

    eixo.set_title(titulo)
    eixo.set_xlabel("Re(z)")
    eixo.set_ylabel("Im(z)")

    eixo.legend(loc="upper right", fontsize=8)
    eixo.grid(True)


# ============================================================
# 6. GRÁFICO DAS SEQUÊNCIAS ELEMENTARES
# ============================================================

indices_plot = [0, 2, 3, 4, 5, 8]

fig, eixos = plt.subplots(2, 3, figsize=(14, 7))

for eixo, i in zip(eixos.ravel(), indices_plot):
    seq = elementares[i]
    x = seq["x"](n_plot)

    eixo.stem(n_plot, x)

    eixo.set_title(f"{seq['tex_x']}\n{seq['tex_X']}   ROC: {seq['roc']}", fontsize=10)
    eixo.set_xlabel("n")
    eixo.set_ylabel("x[n]")
    eixo.grid(True)

plt.suptitle("Sequências elementares e suas Transformadas-Z")
plt.tight_layout()
plt.show()


# ============================================================
# 7. SIGNIFICADO: |X(z)| SOBRE O PLANO COMPLEXO
# ============================================================

# x[n] = a^n u[n]  ->  X(z) = 1 / (1 - a z^-1), ROC: |z| > a
g = np.linspace(-2, 2, 241)
Xg, Yg = np.meshgrid(g, g)
Zg = Xg + 1j * Yg

na_roc = np.abs(Zg) > a + 0.03

with np.errstate(divide="ignore", invalid="ignore"):
    modulo_X = np.where(na_roc, np.abs(1 / (1 - a / Zg)), np.nan)

modulo_X = np.minimum(modulo_X, 6.0)

omega_c = np.linspace(0, 2 * np.pi, 400)
modulo_circulo = np.abs(1 / (1 - a * np.exp(-1j * omega_c)))

fig = plt.figure(figsize=(11, 7))
eixo = fig.add_subplot(projection="3d")

eixo.plot_surface(Xg, Yg, np.ma.masked_invalid(modulo_X), cmap="viridis",
                  linewidth=0, antialiased=False, alpha=0.9)

eixo.plot(np.cos(omega_c), np.sin(omega_c), modulo_circulo, "r", linewidth=2.5,
          label="Círculo unitário (DTFT)")
eixo.plot([a], [0], [6.0], "kx", markersize=10, markeredgewidth=2.5, label="Polo em z = 0.8")

eixo.set_zlim(0, 6)
eixo.set_title(r"$|X(z)|$ para $x[n]=0.8^n\,u[n]$ (apenas dentro da ROC, $|z|>0.8$)")
eixo.set_xlabel("Re(z)")
eixo.set_ylabel("Im(z)")
eixo.set_zlabel("|X(z)|")

eixo.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 8. SIGNIFICADO: X(r e^(jw)) = DTFT DE x[n] r^(-n)
# ============================================================

omega = np.linspace(-np.pi, np.pi, 721)
n_causal = np.arange(0, NM + 1)
x_causal = a ** n_causal

plt.figure(figsize=(11, 5))

for r, cor in zip([0.85, 1.0, 1.5], ["C0", "C1", "C2"]):
    z = r * np.exp(1j * omega)

    fechada = np.abs(1 / (1 - a / z))
    plt.plot(omega, fechada, color=cor, label=f"r = {r}")

    # Conferência com a soma da definição (marcadores)
    direta = np.abs(transformada_z_direta(x_causal, n_causal, z[::45]))
    plt.plot(omega[::45], direta, "o", color=cor, markersize=5)

plt.title(r"$|X(re^{j\omega})|$ para $x[n]=0.8^n u[n]$ (r = 1 é a DTFT; marcadores = soma da definição)")
plt.xlabel(r"$\omega$ (rad/amostra)")
plt.ylabel(r"$|X(re^{j\omega})|$")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 9. REGIÃO DE CONVERGÊNCIA (ROC)
# ============================================================

fig, eixos = plt.subplots(1, 3, figsize=(16, 5.5))

desenhar_plano_z(eixos[0], np.array([a]), np.array([0.0]),
                 lambda m: m > a,
                 r"(a) $0.8^n u[n]$: $|z|>0.8$ (contém $|z|=1$)")

desenhar_plano_z(eixos[1], np.array([a]), np.array([0.0]),
                 lambda m: m < a,
                 r"(b) $-0.8^n u[-n-1]$: $|z|<0.8$ (não contém $|z|=1$)")

desenhar_plano_z(eixos[2], np.array([0.5, 2.0]), np.array([1.25]),
                 lambda m: (m > 0.5) & (m < 2.0),
                 r"(c) $0.5^n u[n]-2^n u[-n-1]$: $0.5<|z|<2$")

plt.suptitle("Regiões de convergência: sequências diferentes podem ter a mesma X(z)")
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


# ============================================================
# 10. ROC NUMÉRICA: SOMAS PARCIAIS DA DEFINIÇÃO
# ============================================================

n_pos = np.arange(0, M_MAX + 1)
n_neg = -np.arange(1, M_MAX + 1)

pontos_z = [1.2, 0.5]
cores_z = ["C0", "C3"]

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

for z0, cor in zip(pontos_z, cores_z):
    fechada = 1 / (1 - a / z0)

    # Sequência causal: a^n u[n], soma de n = 0 até M
    termos_c = (a ** n_pos) * z0 ** (-n_pos.astype(float))
    erro_c = np.abs(np.cumsum(termos_c) - fechada)
    eixos[0].semilogy(n_pos, erro_c, color=cor, label=f"z = {z0}")

    # Sequência anticausal: -a^n u[-n-1], soma de n = -1 até -M
    termos_a = -(a ** n_neg.astype(float)) * z0 ** (-n_neg.astype(float))
    erro_a = np.abs(np.cumsum(termos_a) - fechada)
    eixos[1].semilogy(np.arange(1, M_MAX + 1), erro_a, color=cor, label=f"z = {z0}")

eixos[0].set_title(r"Causal $0.8^n u[n]$ (ROC: $|z|>0.8$)")
eixos[0].set_xlabel("Número de termos somados")

eixos[1].set_title(r"Anticausal $-0.8^n u[-n-1]$ (ROC: $|z|<0.8$)")
eixos[1].set_xlabel("Número de termos somados")

for eixo in eixos:
    eixo.set_ylabel("|soma parcial - 1/(1 - a/z)|")
    eixo.legend()
    eixo.grid(True, which="both")

plt.suptitle("A série converge dentro da ROC e diverge fora dela")
plt.tight_layout()
plt.show()


# ============================================================
# 11. POLOS REAIS E RESPOSTA AO IMPULSO
# ============================================================

n_h = np.arange(0, 41)

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

for p, cor in zip([0.5, 0.9, 1.0, 1.1], ["C0", "C1", "C2", "C3"]):
    h = filtrar(np.array([1.0]), np.array([1.0, -p]), impulso(len(n_h)))
    eixos[0].plot(n_h, h, "o-", color=cor, markersize=3, label=f"p = {p}")

for p, cor in zip([-0.5, -0.9, -1.0, -1.1], ["C0", "C1", "C2", "C3"]):
    h = filtrar(np.array([1.0]), np.array([1.0, -p]), impulso(len(n_h)))
    eixos[1].plot(n_h, h, "o-", color=cor, markersize=3, label=f"p = {p}")

eixos[0].set_title("Polos reais positivos")
eixos[1].set_title("Polos reais negativos (sinal alternado)")

eixos[0].set_ylim(-0.2, 6)
eixos[1].set_ylim(-6, 6)

for eixo in eixos:
    eixo.set_xlabel("n")
    eixo.set_ylabel("h[n] = p^n u[n]")
    eixo.legend()
    eixo.grid(True)

plt.suptitle("H(z) = 1 / (1 - p z^-1): o raio do polo define o crescimento ou decaimento")
plt.tight_layout()
plt.show()


# ============================================================
# 12. POLOS COMPLEXOS: RAIO E ÂNGULO
# ============================================================

def den_par_complexo(r, theta):
    """
    Denominador 1 - 2 r cos(theta) z^-1 + r^2 z^-2
    (polos em r e^(±j theta)).
    """

    return np.array([1.0, -2 * r * np.cos(theta), r ** 2])


n_h = np.arange(0, 61)

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

for r, cor in zip([0.8, 1.0, 1.05], ["C0", "C2", "C3"]):
    h = filtrar(np.array([1.0]), den_par_complexo(r, np.pi / 4), impulso(len(n_h)))
    eixos[0].plot(n_h, h, "o-", color=cor, markersize=3, label=f"r = {r}")

eixos[0].set_title(r"Variando o raio ($\theta=\pi/4$)")

for theta, rotulo, cor in zip([np.pi / 8, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                              [r"$\pi/8$", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$"],
                              ["C0", "C1", "C2", "C4"]):
    h = filtrar(np.array([1.0]), den_par_complexo(0.9, theta), impulso(len(n_h)))
    eixos[1].plot(n_h, h, "o-", color=cor, markersize=3, label=rf"$\theta$ = {rotulo}")

eixos[1].set_title(r"Variando o ângulo (r = 0.9)")

for eixo in eixos:
    eixo.set_xlabel("n")
    eixo.set_ylabel("h[n]")
    eixo.legend()
    eixo.grid(True)

plt.suptitle(r"Polos complexos $re^{\pm j\theta}$: o raio define o decaimento e o ângulo, a oscilação")
plt.tight_layout()
plt.show()


# ============================================================
# 13. SISTEMA DISCRETO: H(z), POLOS, ZEROS E h[n]
# ============================================================

polos_ex, zeros_ex = polos_e_zeros(num_ex, den_ex)
raio_ex = np.max(np.abs(polos_ex))

N_H = 400
n_longo = np.arange(N_H)

h_rec = filtrar(num_ex, den_ex, impulso(N_H))
h_ana = h_analitico(n_longo, num_ex, den_ex)

omega_f = np.linspace(0, np.pi, 400)

H_formula = resposta_em_frequencia(num_ex, den_ex, omega_f)

# DTFT da resposta ao impulso (soma truncada)
H_dtft = np.array([np.sum(h_rec * np.exp(-1j * w * n_longo)) for w in omega_f])

fig, eixos = plt.subplots(1, 3, figsize=(17, 5.5))

desenhar_plano_z(eixos[0], polos_ex, zeros_ex,
                 lambda m: m > raio_ex,
                 r"Polos e zeros de $H(z)$, ROC: $|z|>0.9$")

n_stem = n_longo[:41]
eixos[1].stem(n_stem, h_rec[:41], label="Recursão (eq. de diferenças)")
eixos[1].plot(n_stem, h_ana[:41], "r.", markersize=9, label="Inversa por frações parciais")
eixos[1].set_title("Resposta ao impulso h[n]")
eixos[1].set_xlabel("n")
eixos[1].set_ylabel("h[n]")
eixos[1].legend()
eixos[1].grid(True)

eixos[2].plot(omega_f, np.abs(H_formula), linewidth=2, label=r"$H(z)$ em $z=e^{j\omega}$")
eixos[2].plot(omega_f[::20], np.abs(H_dtft[::20]), "ro", markersize=5, label="DTFT de h[n]")
eixos[2].set_title(r"Resposta em frequência $|H(e^{j\omega})|$")
eixos[2].set_xlabel(r"$\omega$ (rad/amostra)")
eixos[2].set_ylabel("Magnitude")
eixos[2].legend()
eixos[2].grid(True)

plt.suptitle(r"$H(z)=\dfrac{1+0.5z^{-1}}{1-0.9z^{-1}+0.81z^{-2}}$")
plt.tight_layout()
plt.show()


# ============================================================
# 14. SIGNIFICADO DA TRANSFORMADA-Z
# ============================================================

print("TRANSFORMADA-Z")
print("=" * 100)
print("Definição: X(z) = sum( x[n] z^-n ),  z = r e^(j w)")
print("X(z) é a DTFT da sequência x[n] r^-n: o parâmetro r pondera exponencialmente")
print("    a sequência, o que permite analisar também sinais que a DTFT não alcança.")
print("Sobre o círculo unitário (r = 1), X(e^(jw)) é a própria DTFT de x[n].")
print("A soma só converge para alguns valores de z: a Região de Convergência (ROC).")


# ============================================================
# 15. TRANSFORMADAS DE SEQUÊNCIAS ELEMENTARES
# ============================================================

print("\n\nTRANSFORMADAS DE SEQUÊNCIAS ELEMENTARES (a = 0.8, w0 = pi/4)")
print("=" * 125)
print(f"{'Sequência x[n]':<24}{'X(z)':<56}{'ROC':<12}{'Erro |direta - fechada|':>26}")

for seq in elementares:
    x = seq["x"](n_dir)

    X_direta = transformada_z_direta(x, n_dir, seq["z0"])[0]
    X_fechada = seq["X"](np.asarray(seq["z0"], dtype=complex))

    erro = np.abs(X_direta - X_fechada)

    print(f"{seq['nome']:<24}{seq['Xtxt']:<56}{seq['roc']:<12}{erro:26.3e}")

print("\n(Erro avaliado em um ponto z0 dentro da ROC de cada sequência.)")


# ============================================================
# 16. ROC E SEQUÊNCIAS COM A MESMA X(z)
# ============================================================

print("\n\nROC: a mesma X(z) = 1 / (1 - 0.8 z^-1) para duas sequências")
print("=" * 70)

x_causal_c = elementares[3]["x"](n_dir)
x_anticausal = elementares[4]["x"](n_dir)

for z0 in [1.2, 0.5]:
    fechada = 1 / (1 - a / z0)

    soma_c = np.real(transformada_z_direta(x_causal_c[n_dir >= 0][:61], n_dir[n_dir >= 0][:61], z0)[0])
    soma_a = np.real(transformada_z_direta(x_anticausal[n_dir < 0][-60:], n_dir[n_dir < 0][-60:], z0)[0])

    print(f"z = {z0}:  fórmula = {fechada:8.4f}   "
          f"soma causal (60 termos) = {soma_c:12.4g}   soma anticausal (60 termos) = {soma_a:12.4g}")

print("Em |z| = 1.2 só a causal converge; em |z| = 0.5 só a anticausal converge.")


# ============================================================
# 17. POLOS, ZEROS E ESTABILIDADE DE SISTEMAS DISCRETOS
# ============================================================

print("\n\nPOLOS E ESTABILIDADE (h[n] por recursão, 200 amostras)")
print("=" * 90)
print(f"{'Sistema':<28}{'Polos':<26}{'max|p|':>8}{'Estabilidade':>15}{'sum|h[n]|':>14}")

sistemas_polos = [
    ("polo real 0.5", [0.5]),
    ("polo real 0.9", [0.9]),
    ("polo real 1.0", [1.0]),
    ("polo real 1.1", [1.1]),
    ("polo real -0.8", [-0.8]),
    ("par 0.9 e^(±j pi/3)", [0.9 * np.exp(1j * np.pi / 3), 0.9 * np.exp(-1j * np.pi / 3)]),
    ("par 1.05 e^(±j pi/4)", [1.05 * np.exp(1j * np.pi / 4), 1.05 * np.exp(-1j * np.pi / 4)]),
]

for nome, polos in sistemas_polos:
    den = np.real(np.poly(polos))
    h = filtrar(np.array([1.0]), den, impulso(200))

    texto = ", ".join(
        f"{p.real:+.2f}" if abs(p.imag) < 1e-9 else f"{abs(p):.2f}∠±{np.degrees(np.angle(p)):.0f}°"
        for p in np.array(polos, dtype=complex) if p.imag >= -1e-9
    )

    classe = classificar_estabilidade(polos)

    print(f"{nome:<28}{texto:<26}{np.max(np.abs(polos)):8.3f}{classe:>15}{np.sum(np.abs(h)):14.4g}")

print("\nSistema causal é estável quando a ROC |z| > max|p_k| contém o círculo unitário.")

print("\n\nSistema de exemplo: H(z) = (1 + 0.5 z^-1) / (1 - 0.9 z^-1 + 0.81 z^-2)")
print("=" * 70)
print("Equação de diferenças: y[n] = x[n] + 0.5 x[n-1] + 0.9 y[n-1] - 0.81 y[n-2]")

print("Polos: " + ", ".join(f"{p.real:+.3f}{p.imag:+.3f}j" for p in polos_ex)
      + f"   (|p| = {raio_ex:.3f})")
print("Zeros: " + ", ".join(f"{z.real:+.3f}{z.imag:+.3f}j" for z in zeros_ex))
print(f"Estabilidade: {classificar_estabilidade(polos_ex)}")


# ============================================================
# 18. VERIFICAÇÃO DAS PROPRIEDADES E DAS INVERSAS
# ============================================================

print("\n\nVerificações numéricas")
print("=" * 70)

# Inversa: recursão x frações parciais
print(f"h[n]: recursão x frações parciais        erro máximo = {np.max(np.abs(h_rec - h_ana)):.3e}")

# H(z) em z = e^(jw) x DTFT de h[n]
print(f"H(e^jw): fórmula x DTFT de h[n]          erro máximo = {np.max(np.abs(H_formula - H_dtft)):.3e}")

# Deslocamento no tempo: Z{x[n-k]} = z^-k X(z)
x_fin = x_finita(n_dir)
z_teste = 0.9 * np.exp(0.5j)
k_atraso = 2

x_atrasado = x_finita(n_dir - k_atraso)
lado_esq = transformada_z_direta(x_atrasado, n_dir, z_teste)[0]
lado_dir = z_teste ** (-k_atraso) * transformada_z_direta(x_fin, n_dir, z_teste)[0]

print(f"Deslocamento no tempo (k = 2)            erro = {np.abs(lado_esq - lado_dir):.3e}")

# Convolução no tempo = produto das transformadas
x1 = np.array([1.0, 2.0, 3.0])
x2 = np.array([1.0, -1.0, 0.5])
x_conv = np.convolve(x1, x2)

X1 = transformada_z_direta(x1, np.arange(len(x1)), z_teste)[0]
X2 = transformada_z_direta(x2, np.arange(len(x2)), z_teste)[0]
X_conv = transformada_z_direta(x_conv, np.arange(len(x_conv)), z_teste)[0]

print(f"Convolução: Z{{x1*x2}} x X1(z)X2(z)        erro = {np.abs(X_conv - X1 * X2):.3e}")

# Saída do sistema: recursão x convolução com h[n]
n_ent = np.arange(N_RESP)
x_ent = np.cos(0.2 * np.pi * n_ent)

y_rec = filtrar(num_ex, den_ex, x_ent)
y_conv = np.convolve(x_ent, h_rec)[:N_RESP]

print(f"Saída: equação de diferenças x x*h       erro máximo = {np.max(np.abs(y_rec - y_conv)):.3e}")


# ============================================================
# 19. CONCLUSÕES
# ============================================================

print("\n\nCONCLUSÕES")
print("=" * 70)
print("Polos e zeros: raízes do denominador e do numerador de X(z) ou H(z).")
print("O polo p_k contribui com o termo p_k^n na resposta ao impulso:")
print("|p| < 1 decai, |p| = 1 não decai, |p| > 1 cresce.")
print("O ângulo do polo define a frequência de oscilação; o sinal alternado")
print("aparece para polos reais negativos.")
print("Zeros moldam a resposta em frequência (atenuam perto de seu ângulo).")
print("ROC:")
print("Sequência direita (causal): |z| > maior |p|. Esquerda: |z| < menor |p|.")
print("Bilateral: anel entre dois polos. Duração finita: todo z (exceto 0 e/ou infinito).")
print("A ROC faz parte da transformada: X(z) igual com ROCs diferentes = sequências diferentes.")
print("Sistemas discretos:")
print("H(z) = Y(z)/X(z); convolução no tempo vira produto em z.")
print("Atraso de k amostras vira multiplicação por z^-k.")
print("H(e^jw) é H(z) sobre o círculo unitário (resposta em frequência).")
print("Sistema causal é estável (BIBO) se, e somente se, todos os polos estão")
print("dentro do círculo unitário, isto é, a ROC contém |z| = 1.")

print("\nSimulação concluída!")